# Logistic Regression and Classification

Notebook for **Chapter 4** of the *ml* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ml/ch04.html

Tip: change a number, predict the output, then run the cell again.

## 4.1 Sigmoid, Probability, Odds and Log-Odds

## 4.2 The Loss and Its Gradient, Derived

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss

sig = lambda z: 1 / (1 + np.exp(-z))
class MyLogReg:
    def __init__(s, lam=0., lr=0.5, steps=5000, cw=None):
        s.lam, s.lr, s.steps, s.cw = lam, lr, steps, cw      # cw = (weight of class 0, weight of class 1)
    def fit(s, X, y):
        n, p = X.shape; s.w = np.zeros(p); s.b = 0.
        sw = np.ones(n) if s.cw is None else np.where(y == 1, s.cw[1], s.cw[0])
        for _ in range(s.steps):
            pr = sig(X @ s.w + s.b)
            r = sw * (pr - y) / n                              # (p - y), averaged, optionally weighted
            s.w -= s.lr * (X.T @ r + s.lam * s.w)              # gradient + L2 term; the intercept is not penalized
            s.b -= s.lr * r.sum()
        return s
    def predict_proba(s, X): return sig(X @ s.w + s.b)

d = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(d.data, d.target, test_size=.25, random_state=0, stratify=d.target)
sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
m = MyLogReg(lam=0.01).fit(Ztr, ytr)
n = len(ytr)
sk = LogisticRegression(C=1/(n*0.01), tol=1e-10, max_iter=10000).fit(Ztr, ytr)
print("max |w diff|", np.abs(m.w - sk.coef_[0]).max(), "b", m.b, sk.intercept_)
pm = m.predict_proba(Zte)
print("test acc", accuracy_score(yte, pm > .5), accuracy_score(yte, sk.predict(Zte)))
print("logloss", log_loss(yte, pm), log_loss(yte, sk.predict_proba(Zte)[:, 1]))

## 4.3 Decision Boundary, Thresholds and Class Weights

In [ ]:
from sklearn.datasets import make_classification
from sklearn.metrics import precision_score, recall_score
X, yy = make_classification(5000, 10, n_informative=4, weights=[.95], flip_y=.02, random_state=0, class_sep=1.0)
Xa, Xb, ya, yb = train_test_split(X, yy, test_size=.3, random_state=0, stratify=yy)
for cw in [None, "balanced"]:
    c = LogisticRegression(class_weight=cw, max_iter=1000).fit(Xa, ya); pr = c.predict(Xb)
    print("cw", cw, "acc", round(accuracy_score(yb, pr), 3), "prec", round(precision_score(yb, pr), 3),
          "recall", round(recall_score(yb, pr), 3), "mean p", round(c.predict_proba(Xb)[:, 1].mean(), 3))

prevalence 0.0574
cw None acc 0.944 prec 0.667 recall 0.047 mean p 0.061
cw balanced acc 0.679 prec 0.101 recall 0.581 mean p 0.411


## 4.4 Many Classes: Softmax

## 4.5 Generalized Linear Models: Counts, Money and Exposure

In [ ]:
from sklearn.linear_model import PoissonRegressor, GammaRegressor, LinearRegression
rng = np.random.default_rng(0); n = 20000
age = rng.uniform(18, 70, n); urban = rng.integers(0, 2, n); expo = rng.uniform(.2, 1, n)
rate = np.exp(-3 + 0.02*(age - 40) + 0.5*urban); cnt = rng.poisson(rate * expo)
X = np.c_[age - 40, urban]
pr = PoissonRegressor(alpha=0, max_iter=1000).fit(X, cnt/expo, sample_weight=expo)
print("Poisson coef", pr.coef_, "intercept", pr.intercept_, "true [0.02,0.5,-3]")
print("mean count", cnt.mean(), "var", cnt.var())
print("OLS on rate", LinearRegression().fit(X, cnt/expo, sample_weight=expo).coef_)

Poisson coef [0.0207 0.5838] intercept -3.06034893616742 true [0.02,0.5,-3]
mean count 0.04485 var 0.0466384775
OLS on rate [0.0015 0.0425]


## 4.6 The Perceptron, the XOR Problem and the Bridge to Neural Networks

In [ ]:
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
Xx = np.array([[0,0],[0,1],[1,0],[1,1.]]); yx = np.array([0,1,1,0])
pc = Perceptron(max_iter=1000, tol=None, random_state=0).fit(Xx, yx); print("perceptron XOR acc", pc.score(Xx, yx), pc.predict(Xx))
Xx2 = np.c_[Xx, Xx[:,0]*Xx[:,1]]; lr = LogisticRegression(C=1e4).fit(Xx2, yx); print("logreg w/ x1*x2 acc", lr.score(Xx2, yx))
print("perceptron AND", Perceptron(max_iter=1000, tol=None, random_state=0).fit(Xx, np.array([0,0,0,1])).score(Xx, np.array([0,0,0,1])))
mlp = MLPClassifier((4,), activation="tanh", solver="lbfgs", random_state=1, max_iter=2000).fit(Xx, yx); print("MLP XOR", mlp.predict(Xx))

perceptron XOR acc 0.5 [0 0 0 0]
logreg w/ x1*x2 acc 1.0
perceptron AND 1.0
MLP XOR [0 1 1 0]


## Chapter recap

## Practice set

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ml/ch04.html